# 06 指标计算 (core.metrics)

基于 `hscredit.xlsx` 真实信贷数据，演示所有风控指标计算。

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import hscredit as hsc
from hscredit import (init_setting, WOEEncoder, LogisticRegression, OptimalBinning)
from hscredit.core.metrics import (
    ks, auc, gini, accuracy, precision, recall, f1,
    ks_bucket, iv, iv_table,
    psi, psi_table, psi_rating, csi, batch_psi,
    mse, mae, rmse, r2,
)
from hscredit.core.metrics.finance import lift_at, lift_monotonicity_check, rule_lift_table
from hscredit import KS, AUC, Gini, PSI, IV, PSI_table, IV_table
init_setting()

df = pd.read_excel('hscredit.xlsx')
df['loan_date'] = pd.to_datetime(df['loan_date'], unit='D', origin='1899-12-30')
target = 'FPD15'
features = ['bj_qy24','mobtech80','bairong_v1','lender_count_12m','lender_count_6m',
    'overdue_loan_m1_count_12m','loan_count_12m','loan_amt_sum_12m',
    'network_loan_lender_count','last_performance_days']
X = df[features].fillna(-999); y = df[target]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
woe = WOEEncoder(max_n_bins=8)
X_woe_tr = woe.fit_transform(X_tr, y_tr)
X_woe_te = woe.transform(X_te)
lr = LogisticRegression(C=0.1, max_iter=1000); lr.fit(X_woe_tr, y_tr)
y_prob = lr.predict_proba(X_woe_te)[:,1]
print('模型训练完成')

## 1. 分类指标 (KS / AUC / Gini)

In [ ]:
print(f'KS:        {ks(y_te, y_prob):.4f}')
print(f'AUC:       {auc(y_te, y_prob):.4f}')
print(f'Gini:      {gini(y_te, y_prob):.4f}')
y_pred = (y_prob > 0.15).astype(int)  # 信贷场景低阈值
print(f'Precision: {precision(y_te, y_pred):.4f}')
print(f'Recall:    {recall(y_te, y_pred):.4f}')
print(f'F1:        {f1(y_te, y_pred):.4f}')

In [ ]:
print('\nKS分箱表:')
ks_bucket(y_te.values, y_prob, n=10)

## 2. 特征评估 (IV / WOE)

In [ ]:
feat = df['bj_qy24'].fillna(-999).values
print(f'bj_qy24 IV: {iv(y.values, feat):.4f}')
print('\nIV详细表:')
iv_table(y.values, feat, method='quantile', max_n_bins=8)[['分箱标签','样本总数','坏样本率','分档WOE值','分档IV值']]

## 3. 稳定性 (PSI / CSI)

In [ ]:
base = X_tr['bj_qy24'].values
actual = X_te['bj_qy24'].values
psi_val = psi(base, actual, max_n_bins=8)
print(f'bj_qy24 PSI: {psi_val:.4f}')
print(f'稳定性评级: {psi_rating(psi_val)}')
print()
# PSI详细表
psi_table(base, actual, max_n_bins=8)

In [ ]:
# 批量PSI
batch_result = batch_psi(X_tr, X_te, features=features[:6])
print('批量PSI:')
print(batch_result)

## 4. 金融风控指标 (Lift)

In [ ]:
print(f'Lift@5%:  {lift_at(y_te.values, y_prob, ratios=0.05):.4f}')
print(f'Lift@10%: {lift_at(y_te.values, y_prob, ratios=0.10):.4f}')
print(f'Lift@20%: {lift_at(y_te.values, y_prob, ratios=0.20):.4f}')
print(f'Lift@30%: {lift_at(y_te.values, y_prob, ratios=0.30):.4f}')
print(f'单调性检查: {lift_monotonicity_check(y_te.values, y_prob)}')

# 规则效果评估
rule_hit = y_prob > 0.2
print('\n规则命中分析:')
rule_lift_table(y_te.values, rule_hit)

## 5. 大写别名兼容（历史API）

In [ ]:
print('KS (大写):', KS(y_te, y_prob))
print('AUC (大写):', AUC(y_te, y_prob))
print('Gini (大写):', Gini(y_te, y_prob))
print('PSI (大写):', PSI(base, actual))
print('IV (大写):', IV(y.values, feat))

## 6. 训练/测试集指标对比

In [ ]:
y_prob_tr = lr.predict_proba(X_woe_tr)[:,1]
print(f'指标对比:')
pd.DataFrame({
    '数据集': ['训练集', '测试集'],
    'KS': [round(ks(y_tr,y_prob_tr),4), round(ks(y_te,y_prob),4)],
    'AUC': [round(auc(y_tr,y_prob_tr),4), round(auc(y_te,y_prob),4)],
    'Gini': [round(gini(y_tr,y_prob_tr),4), round(gini(y_te,y_prob),4)],
    '坏率': [round(y_tr.mean(),4), round(y_te.mean(),4)],
})